# Paggawa ng Mga Application sa Pagbuo ng Imahe (OpenAI)

Hindi lamang text generation ang kaya ng LLMs. Maaari ka ring gumawa ng mga larawan mula sa mga paglalarawan sa teksto. Ang mga larawan bilang isang modality ay kapaki-pakinabang sa MedTech, arkitektura, turismo, pag-develop ng laro, marketing, at iba pa. Sa araling ito gagamit tayo ng mga kasalukuyang **GPT Image** na mga modelo sa OpenAI platform.

## Mga Layunin sa Pagkatuto

Sa pagtatapos ng araling ito ay magiging kaya mong:

- Ipaliwanag kung ano ang image generation at saan ito kapaki-pakinabang.
- Maunawaan ang pamilyang modelo ng `gpt-image` at kung paano ito naiiba sa legacy na mga DALL·E na modelo.
- Bumuo ng isang application sa pagbuo ng imahe at mag-edit ng mga imahe.

## Ano ang image generation?

Ang mga modelo sa pagbuo ng imahe ay lumilikha ng mga larawan mula sa isang text prompt. Ang mga makabagong modelo tulad ng `gpt-image` ay natututo ng ugnayan sa pagitan ng teksto at mga larawan habang nagsasanay, pagkatapos ay sunud-sunod nitong binabago ang random na ingay upang maging isang larawan na tumutugma sa iyong prompt.

Dalawang kilalang pamilyang mga modelo ng imahe ay:

- **`gpt-image` (OpenAI)** — ang kasalukuyang henerasyon na ginagamit sa araling ito. Sinusuportahan nito ang text-to-image generation at pag-edit ng imahe (inpainting gamit ang mask).
- **Midjourney** — isang popular na third-party na modelo na may sariling serbisyo at workflow na naka-Discord.

> Ang mga lumang OpenAI na modelo ng imahe — **DALL·E 2** at **DALL·E 3** — ay legacy na. Hindi na available ang DALL·E 3 para sa mga bagong deployment, at ang mga tampok tulad ng `create_variation` ay mayroon lamang sa DALL·E 2. Gamitin ang mga `gpt-image` na mga modelo para sa mga bagong application.

> **Mahalaga:** Ang mga `gpt-image` na modelo ay nagbabalik ng generated na imahe bilang **base64** (`b64_json`), hindi bilang isang URL. Ina-decode ng iyong code ang base64 string sa bytes at sine-save ito — walang image URL na maaaring i-download.


## Pagbuo ng iyong unang application para sa paglikha ng imahe

Ano nga ba ang kailangan upang makabuo ng isang application para sa paglikha ng imahe? Kailangan mo ang mga sumusunod na library:

- **python-dotenv**, lubos na inirerekomenda na gamitin ang library na ito upang itago ang iyong mga lihim sa isang *.env* na file na hiwalay sa code.
- **openai**, ito ang library na gagamitin mo para makipag-ugnayan sa OpenAI API.
- **pillow**, para sa pagtatrabaho sa mga imahe sa Python.


1. Gumawa ng isang file na *.env* na may sumusunod na nilalaman:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Kolektahin ang mga nasabing mga librarya sa isang file na tinatawag na *requirements.txt* tulad nito:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Susunod, lumikha ng virtual na kapaligiran at i-install ang mga librarya:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Para sa Windows, gamitin ang sumusunod na mga utos para gumawa at i-activate ang iyong virtual environment:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Idagdag ang sumusunod na code sa isang file na tinatawag na *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Create OpenAI object (reads OPENAI_API_KEY from your .env)
    client = openai.OpenAI()


    try:
        # Create an image by using the image generation API
        generation_response = client.images.generate(
            model="gpt-image-2",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
            size='1024x1024',
            n=1
        )
        # Set the directory for the stored image
        image_dir = os.path.join(os.curdir, 'images')

        # If the directory doesn't exist, create it
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialize the image path (note the filetype should be png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image models return the image as base64 (b64_json), not a URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Display the image in the default image viewer
        image = Image.open(image_path)
        image.show()

    # catch exceptions
    except openai.BadRequestError as err:
        print(err)

    ```

Ipaliwanag natin ang code na ito:

- Una, ini-import natin ang mga library na kailangan natin, kasama na ang OpenAI library, dotenv library, base64 module, at Pillow library.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Pagkatapos nito, gumawa tayo ng client, na bumabasa ng API key mula sa iyong ``.env``.

    ```python
    # Create OpenAI object
    client = openai.OpenAI()
    ```

- Sunod, gumawa tayo ng imahe:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-2",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` models ay nagbabalik ng imahe bilang isang **base64** string sa loob ng `data[0].b64_json`. Dinidecode natin ito sa bytes at sinusulat sa isang file — walang URL na ida-download.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Panghuli, binubuksan natin ang imahe at ginagamit ang karaniwang image viewer para ipakita ito:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Karagdagang detalye sa paggawa ng imahe

Tingnan natin ang mga parameter ng `images.generate`:

- **model**, ang image model na gagamitin (halimbawa `gpt-image-2`).
- **prompt**, ang text prompt na ginamit para gumawa ng imahe. Dito ay "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, ang sukat ng nagawang imahe (halimbawa `1024x1024`, `1536x1024`, `1024x1536`, o `"auto"`).
- **n**, ang bilang ng mga imaheng ginawa. Dito, gumawa tayo ng isa.

> Hindi gumagamit ang mga image model ng parameter na `temperature` — ito ay para sa text-generation control. Para makakuha ng iba-ibang resulta, tawagin muli ang API; para mabawasan ang iba-ibang resulta, gawing mas tiyak ang iyong prompt.

## Karagdagang kakayahan sa paggawa ng imahe

Nakita mo kung paano gumawa ng imahe gamit ang iilang linya ng Python. Ang `gpt-image` models ay maaari ring **mag-edit** ng umiiral na imahe. Sa pamamagitan ng pagbibigay ng imahe, opsyonal na **mask** (na nagmamarka ng bahaging babaguhin), at isang prompt, maaari mong baguhin ang bahagi ng imahe — halimbawa, magdagdag ng sumbrero sa ating kuneho.

```python
response = client.images.edit(
  model="gpt-image-2",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# edits are also returned as base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Ang base image ay naglalaman lamang ng kuneho; ang huling imahe ay may sumbrero sa kuneho.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
